In [1]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from surprise import Dataset, Reader, SVD
from surprise.model_selection import train_test_split

In [2]:
ratings = pd.read_csv("../data/raw/ratings.csv")
movies = pd.read_csv("../data/raw/movies.csv")

ratings_small = ratings.sample(200000, random_state=42)

movies["genres_clean"] = movies["genres"].str.replace("|", " ", regex=False)
movies_small = movies.head(5000).copy().reset_index(drop=True)

In [3]:
reader = Reader(rating_scale=(0.5, 5.0))

data = Dataset.load_from_df(
    ratings_small[["userId", "movieId", "rating"]],
    reader
)

trainset, testset = train_test_split(data, test_size=0.2, random_state=42)

svd_model = SVD()
svd_model.fit(trainset)

In [4]:
tfidf = TfidfVectorizer(stop_words="english")

tfidf_matrix = tfidf.fit_transform(movies_small["genres_clean"].fillna(""))

cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)

indices = pd.Series(
    movies_small.index,
    index=movies_small["title"]
).drop_duplicates()

In [5]:
def hybrid_recommend(user_id, liked_movie_title=None, top_n=10):
    all_movie_ids = movies_small["movieId"].unique()

    watched_movies = ratings_small[
        ratings_small["userId"] == user_id
    ]["movieId"].unique()

    unwatched_movies = [
        movie_id for movie_id in all_movie_ids
        if movie_id not in watched_movies
    ]

    results = []

    for movie_id in unwatched_movies:
        pred = svd_model.predict(user_id, movie_id)
        svd_score = pred.est / 5.0

        avg_rating = ratings_small[
            ratings_small["movieId"] == movie_id
        ]["rating"].mean()

        if pd.isna(avg_rating):
            collab_score = 0.5
        else:
            collab_score = avg_rating / 5.0

        content_score = 0.5

        if liked_movie_title and liked_movie_title in indices:
            liked_idx = indices[liked_movie_title]

            movie_row = movies_small[movies_small["movieId"] == movie_id]

            if not movie_row.empty:
                movie_idx = movie_row.index[0]
                content_score = cosine_sim[liked_idx][movie_idx]

        final_score = (
            0.4 * svd_score +
            0.3 * collab_score +
            0.3 * content_score
        )

        results.append((movie_id, svd_score, collab_score, content_score, final_score))

    result_df = pd.DataFrame(
        results,
        columns=["movieId", "svd_score", "collab_score", "content_score", "final_score"]
    )

    result_df = result_df.merge(movies_small, on="movieId")

    result_df = result_df.sort_values("final_score", ascending=False)

    return result_df.head(top_n)[
        ["movieId", "title", "genres", "svd_score", "collab_score", "content_score", "final_score"]
    ]

In [6]:
hybrid_recommend(
    user_id=1,
    liked_movie_title="Toy Story (1995)",
    top_n=10
)

,movieId,title,genres,svd_score,collab_score,content_score,final_score
3021,3114,Toy Story 2 (1999),Adventure|Animation|Children|Comedy|Fantasy,0.794811,0.764593,1.000000,0.847302
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,0.755925,0.782143,1.000000,0.837013
4932,5038,"Flight of Dragons, The (1982)",Adventure|Animation|Children|Drama|Fantasy,0.731220,0.850000,0.945387,0.831104
4780,4886,"Monsters, Inc. (2001)",Adventure|Animation|Children|Comedy|Fantasy,0.733294,0.777104,1.000000,0.826449
705,720,Wallace & Gromit: The Best of Aardman Animatio...,Adventure|Animation|Comedy,0.860036,0.857746,0.740828,0.823587
3912,4016,"Emperor's New Groove, The (2000)",Adventure|Animation|Children|Comedy|Fantasy,0.728661,0.765517,1.000000,0.821119
729,745,Wallace & Gromit: A Close Shave (1995),Animation|Children|Comedy,0.811288,0.850685,0.775255,0.812297
2908,3000,Princess Mononoke (Mononoke-hime) (1997),Action|Adventure|Animation|Drama|Fantasy,0.821757,0.786364,0.760373,0.792724
4790,4896,Harry Potter and the Sorcerer's Stone (a.k.a. ...,Adventure|Children|Fantasy,0.814914,0.758768,0.783783,0.788731
4201,4306,Shrek (2001),Adventure|Animation|Children|Comedy|Fantasy|Ro...,0.691442,0.752717,0.945597,0.786071
